In [6]:
import pandas as pd
import keras
from keras.layers import Input, Flatten, Dense, Lambda, Reshape, Activation, ReLU
from keras.models import Model
from keras import backend as K
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, confusion_matrix
import tensorflow as tf
import keras
import keras_tuner
from keras_tuner import HyperModel
from keras_tuner import HyperParameters
from keras_tuner.tuners import BayesianOptimization
from keras_tuner.engine.hyperparameters import HyperParameters

In [11]:
# Cargar el dataset
# Ficheros V1
#file_train = r"C:\Users\Usuario\Desktop\LSTM2\datos\OLD\datos_entrenamiento_serie_sat3.xlsx"
#file_test = r"C:\Users\Usuario\Desktop\LSTM2\datos\OLD\datos_validacion_serie_sat3.xlsx"

# Ficheros V2
file_train = r"C:\Users\Usuario\Desktop\LSTM2\datos\N\N_datos_entrenamiento_serie_sat3.xlsx"
file_test = r"C:\Users\Usuario\Desktop\LSTM2\datos\N\N_datos_validacion_serie_sat3.xlsx"

X_train = pd.read_excel(file_train, sheet_name = "Sheet1", index_col = 0)
X_test = pd.read_excel(file_test, sheet_name = "Sheet1", index_col = 0)

In [12]:
# Preparar los datos de entrenamiento
X_train = X_train[X_train.anomaly == 0]  ## Usaremos únicamente la clase 0 (transacciones normales)
Y_train = X_train['anomaly'] #identificación de si los datos de train son anómalos o no
X_train = X_train.drop(columns = ["anomaly"], axis=1)
X_train_columns = X_train.columns
X_train = X_train.values
X_train = X_train.reshape(1, X_train.shape[0], X_train.shape[1])

# Preparar los datos de test
X_test = X_test[X_test.anomaly == 0]  ## Usaremos únicamente la clase 0 (transacciones normales)
Y_test = X_test['anomaly'] #identificación de si los datos de test son anómalos o no
X_test = X_test.drop(columns = ["anomaly"], axis=1)
X_test_columns = X_test.columns
X_test = X_test.values
X_test = X_test.reshape(1, X_test.shape[0], X_test.shape[1])

In [17]:
#definimos el modelo con los hiperparámetros a optimizar (en este caso, learning rate, batch size, funciones de activación intermedias y función de coste)
def build_model(hp):
    hp_learning_rate = hp.Float('lr', min_value = 0.00001, max_value = 0.001, step = 10, sampling='log')
    hp_batch_size = hp.Int('batch_size', min_value=32, max_value=128, step=2, sampling='log')
    hp_loss_funct = hp.Choice('loss_funct', ['mse', 'mae', 'msle'])
    
    hp_act_funct_out = hp.Choice('act_out', ['linear', 'sigmoid'])
    
    hp_units = hp.Int('units',min_value=32, max_value=128, step=2, sampling='log')
    hp_dropout = hp.Float('dropout', min_value=0, max_value=0.5, step=0.1)

    model = tf.keras.models.Sequential([
      tf.keras.layers.LSTM(hp_units, return_sequences=True),
      tf.keras.layers.Dropout(hp_dropout),
      tf.keras.layers.LSTM(hp_units, return_sequences=True),
      tf.keras.layers.Dropout(hp_dropout),
      tf.keras.layers.Dense(X_train.shape[2], activation = hp_act_funct_out),
    ])

    model.compile(keras.optimizers.Adam(learning_rate=hp_learning_rate), loss=hp_loss_funct)

    return model

In [18]:
build_model(keras_tuner.HyperParameters())

<Sequential name=sequential, built=False>

In [19]:
#iniciamos la optimización bayesiana, con la función objetivo que queremos optimizar (en este caso, la variable val_loss)
tuner = keras_tuner.BayesianOptimization(
    build_model,
    objective = "val_loss",
    project_name='project_name1')

In [20]:
#entrenamos el modelo con el método search
tuner.search(X_train, X_train, epochs=50, validation_data=(X_test, X_test))

Trial 10 Complete [00h 00m 46s]
val_loss: 0.07489138841629028

Best val_loss So Far: 0.0032716465648263693
Total elapsed time: 00h 08m 36s


In [21]:
#visualizamos los hiperparámetros que ha utilizado el modelo
tuner.search_space_summary()

Search space summary
Default search space size: 6
lr (Float)
{'default': 1e-05, 'conditions': [], 'min_value': 1e-05, 'max_value': 0.001, 'step': 10, 'sampling': 'log'}
batch_size (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 128, 'step': 2, 'sampling': 'log'}
loss_funct (Choice)
{'default': 'mse', 'conditions': [], 'values': ['mse', 'mae', 'msle'], 'ordered': False}
act_out (Choice)
{'default': 'linear', 'conditions': [], 'values': ['linear', 'sigmoid'], 'ordered': False}
units (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 128, 'step': 2, 'sampling': 'log'}
dropout (Float)
{'default': 0.0, 'conditions': [], 'min_value': 0.0, 'max_value': 0.5, 'step': 0.1, 'sampling': 'linear'}


In [22]:
#visualizamos los mejores resultados
tuner.results_summary()

Results summary
Results in .\project_name1
Showing 10 best trials
Objective(name="val_loss", direction="min")

Trial 02 summary
Hyperparameters:
lr: 0.001
batch_size: 128
loss_funct: msle
act_out: sigmoid
units: 128
dropout: 0.2
Score: 0.0032716465648263693

Trial 08 summary
Hyperparameters:
lr: 0.0001
batch_size: 128
loss_funct: mae
act_out: linear
units: 128
dropout: 0.0
Score: 0.06709398329257965

Trial 09 summary
Hyperparameters:
lr: 0.0001
batch_size: 64
loss_funct: mse
act_out: linear
units: 64
dropout: 0.30000000000000004
Score: 0.07489138841629028

Trial 03 summary
Hyperparameters:
lr: 1e-05
batch_size: 128
loss_funct: msle
act_out: linear
units: 128
dropout: 0.0
Score: 0.07937726378440857

Trial 07 summary
Hyperparameters:
lr: 1e-05
batch_size: 64
loss_funct: msle
act_out: linear
units: 32
dropout: 0.30000000000000004
Score: 0.10056918114423752

Trial 06 summary
Hyperparameters:
lr: 1e-05
batch_size: 128
loss_funct: msle
act_out: sigmoid
units: 64
dropout: 0.4
Score: 0.1062640

In [23]:
#obtenemos el mejor modelo, y el listado de los hiperparámetros óptimos
best_model = tuner.get_best_models(num_models=1)[0]
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_hps.values

C:\Users\Usuario\anaconda3\Lib\site-packages\keras\src\saving\saving_lib.py:713: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


{'lr': 0.001,
 'batch_size': 128,
 'loss_funct': 'msle',
 'act_out': 'sigmoid',
 'units': 128,
 'dropout': 0.2}